# Process and save CS calculated using the results of Trendy (v13).


#### Libraries

In [1]:
from data_compilation import *
from figures_functions import *

### Read data

In [2]:
d1='2001-01'; d2='2014-12'

In [6]:
nbp=get_nbpTrendy(d1,d2)
nbp=nbp.drop_vars(["CARDAMOM"])
nep=get_nepTrendy(d1,d2)[0]
nep=nep.drop_vars(["CARDAMOM"])
nep,gpp,ra,rh=get_nepTrendy(d1,d2)
nep=nep.drop_vars(["CARDAMOM"]); gpp=gpp.drop_vars(["CARDAMOM"]); ra=ra.drop_vars(["CARDAMOM"]); rh=rh.drop_vars(["CARDAMOM"])

# weight of each model.
wm_nep=xr.open_dataset('/home/data/ResultsCS/Weights/nepTRENDY_ob_FLUXCOM-X.nc')
wm_nbp=xr.open_dataset('/home/data/ResultsCS/Weights/nbpTRENDY_ob_CarboScope.nc')

CABLE-POP
CARDAMOM
CLASSIC
CLM5.0
ED
ELM


/home/estefania/cmip6stop/CS/data_compilation.py:399: UserWarning: rename 'lat' to 'latitude' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  nbp=nbp.rename(lat="latitude",lon='longitude')
/home/estefania/cmip6stop/CS/data_compilation.py:399: UserWarning: rename 'lon' to 'longitude' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  nbp=nbp.rename(lat="latitude",lon='longitude')


IBIS
iMAPLE
ISAM
ISBACTRIP
JSBACH
JULES
LPJwsl
LPX
OCN
ORCHIDEE
SDGVM
VISIT
VISIT-UT


In [8]:
nep["time"]=nep["time"].astype("datetime64[ns]")
gpp["time"]=gpp["time"].astype("datetime64[ns]")
ra["time"]=ra["time"].astype("datetime64[ns]")
rh["time"]=rh["time"].astype("datetime64[ns]")
nbp["time"]=nbp["time"].astype("datetime64[ns]")

#### Only the common period with other products

In [2]:
d1='2001-01'; d2='2014-12'
nep=nep.sel(time=slice(d1,d2))
nbp=nbp.sel(time=slice(d1,d2))

#### Means

In [5]:
nep_mean=nep.mean("time")
gpp_mean=gpp.mean("time")
ra_mean=ra.mean("time")
rh_mean=rh.mean("time")
nbp_mean=nbp.mean("time")
gc.collect()

0

### CS

In [6]:
Mt_nep=nep.cumulative_integrate("time",datetime_unit='D')/30.4/12 # mass [kg C m-2]
cs_nep=Mt_nep.integrate("time",datetime_unit='D')/30.4/12 # carbon sequestration [kg C m-2 yr] # without substracting the initial value. 

# Mt_nbp=nbp.cumulative_integrate("time",datetime_unit='D')/30.4/12 # mass [kg C m-2]
# cs_nbp=Mt_nbp.integrate("time",datetime_unit='D')/30.4/12 # carbon sequestration [kg C m-2 yr] # without substracting the initial value.


### Ensembles


#### Using the weight maps

In [7]:
# NEP
Mtnep_w=Mt_nep*wm_nep
csnep_w=cs_nep*wm_nep
mean=Mtnep_w.to_array(dim='new').sum('new')*maskl; Mt_nep=Mt_nep.assign(mmean=mean)
mean=csnep_w.to_array(dim='new').sum('new')*maskl; cs_nep=cs_nep.assign(mmean=mean)
# Means
gpp_mean_w=gpp_mean*wm_nep
ra_mean_w=ra_mean*wm_nep; rh_mean_w=rh_mean*wm_nep
nep_mean_w=nep_mean*wm_nep
mean=gpp_mean_w.to_array(dim='new').sum('new')*maskl; gpp_mean=gpp_mean.assign(mmean=mean)
mean=ra_mean_w.to_array(dim='new').sum('new')*maskl; ra_mean=ra_mean.assign(mmean=mean)
mean=rh_mean_w.to_array(dim='new').sum('new')*maskl; rh_mean=rh_mean.assign(mmean=mean)

Mt_nep=Mt_nep.assign_attrs(units="kgC m-2", description="Carbon mass calculated with NEP from TRENDY models", Ensamble="The mean value was calculated with the weights of each model obtained by comparing the nep values of each model with those of FLUXCOM-X.")
cs_nep=cs_nep.assign_attrs(units="kgC m-2 yr", description="Carbon sequestration calculated with NEP from TRENDY models", Ensamble="The mean value was calculated with the weights of each model obtained by comparing the nep values of each model with those of FLUXCOM-X.")
nep_mean=nep_mean.assign_attrs(units="kgC m-2 yr-1", description="NEP calculated with GPP, ra and rh from TRENDYmodels", Ensamble="The mean value was calculated with the weights of each model obtained by comparing the nep values of each model with those of FLUXCOM-X.")
gpp_mean=gpp_mean.assign_attrs(units="kgC m-2 yr-1", description="GPP from TRENDY models", Ensamble="The mean value was calculated with the weights of each model obtained by comparing the nep values of each model with those of FLUXCOM-X.")
ra_mean=ra_mean.assign_attrs(units="kgC m-2 yr-1", description="Autotrophic respiration from TRENDY models", Ensamble="The mean value was calculated with the weights of each model obtained by comparing the nep values of each model with those of FLUXCOM-X.")
rh_mean=rh_mean.assign_attrs(units="kgC m-2 yr-1", description="Heterotrophic respiration from TRENDY models", Ensamble="The mean value was calculated with the weights of each model obtained by comparing the nep values of each model with those of FLUXCOM-X.")

# NBP
Mtnbp_w=Mt_nbp*wm_nbp
csnbp_w=cs_nbp*wm_nbp
mean=Mtnbp_w.to_array(dim='new').sum('new')*maskl; Mt_nbp=Mt_nbp.assign(mmean=mean)
mean=csnbp_w.to_array(dim='new').sum('new')*maskl; cs_nbp=cs_nbp.assign(mmean=mean)

# Means
nbp_mean_w=nbp_mean*wm_nbp
mean=nbp_mean_w.to_array(dim='new').sum('new')*maskl; nbp_mean=nbp_mean.assign(mmean=mean)

Mt_nbp=Mt_nbp.assign_attrs(units="kgC m-2", description="Carbon mass calculated with NBP from TRENDY models", Ensamble="The mean value was calculated with the weights of each model obtained by comparing the nbp values of each model with those of CarboSope.")
cs_nbp=cs_nbp.assign_attrs(units="kgC m-2 yr", description="Carbon sequestration calculated with NBP from TRENDY models", Ensamble="The mean value was calculated with the weights of each model obtained by comparing the nbp values of each model with those of CarboSope.")
nbp_mean=nbp_mean.assign_attrs(units="kgC m-2 yr-1", description="NBP from TRENDY models", Ensamble="The mean value was calculated with the weights of each model obtained by comparing the nbp values of each model with those of CarboScope.")


In [11]:
gc.collect()

Mt_nep.to_netcdf('/home/data/ResultsCS/MT_nep_TRENDY_w.nc')
cs_nep.to_netcdf('/home/data/ResultsCS/CS_nep_TRENDY_w.nc')
Mt_nep.to_netcdf('/home/data/ResultsCS/MT_nep_TRENDY_w_2001-2014.nc')
cs_nep.to_netcdf('/home/data/ResultsCS/CS_nep_TRENDY_w_2001-2014.nc')
gpp_mean.to_netcdf('/home/data/ResultsCS/gpp_mean_TRENDY_w.nc')
ra_mean.to_netcdf('/home/data/ResultsCS/ra_mean_TRENDY_w.nc')
rh_mean.to_netcdf('/home/data/ResultsCS/rh_mean_TRENDY_w.nc')
nep_mean.to_netcdf('/home/data/ResultsCS/nep_mean_TRENDY_w.nc')


Mt_nbp.to_netcdf('/home/data/ResultsCS/MT_nbp_TRENDY_w.nc')
cs_nbp.to_netcdf('/home/data/ResultsCS/CS_nbp_TRENDY_w.nc')
nbp_mean.to_netcdf('/home/data/ResultsCS/nbp_mean_TRENDY_w.nc')

Mt_nbp.to_netcdf('/home/data/ResultsCS/MT_nbp_TRENDY_w_2001-2014.nc')
cs_nbp.to_netcdf('/home/data/ResultsCS/CS_nbp_TRENDY_w_2001-2014.nc')


#### All models having the same weight

In [9]:
# all data
mean=gpp.to_array(dim='new').mean('new')*maskls.LSMASK; gpp=gpp.assign(mmean=mean)
mean=ra.to_array(dim='new').mean('new')*maskls.LSMASK; ra=ra.assign(mmean=mean)
mean=rh.to_array(dim='new').mean('new')*maskls.LSMASK; rh=rh.assign(mmean=mean)
mean=nep.to_array(dim='new').mean('new')*maskls.LSMASK; nep=nep.assign(mmean=mean)
mean=nbp.to_array(dim='new').mean('new')*maskls.LSMASK; nbp=nbp.assign(mmean=mean)
mean=Mt_nep.to_array(dim='new').mean('new')*maskls.LSMASK; Mt_nep=Mt_nep.assign(mmean=mean)
mean=Mt_nbp.to_array(dim='new').mean('new')*maskls.LSMASK; Mt_nbp=Mt_nbp.assign(mmean=mean)
mean=cs_nep.to_array(dim='new').mean('new')*maskls.LSMASK; cs_nep=cs_nep.assign(mmean=mean)
mean=cs_nbp.to_array(dim='new').mean('new')*maskls.LSMASK; cs_nbp=cs_nbp.assign(mmean=mean)

# means
mean=gpp_mean.to_array(dim='new').mean('new')*maskls.LSMASK; gpp_mean=gpp_mean.assign(mmean=mean)
mean=ra_mean.to_array(dim='new').mean('new')*maskls.LSMASK; ra_mean=ra_mean.assign(mmean=mean)
mean=rh_mean.to_array(dim='new').mean('new')*maskls.LSMASK; rh_mean=rh_mean.assign(mmean=mean)
mean=nep_mean.to_array(dim='new').mean('new')*maskls.LSMASK; nep_mean=nep_mean.assign(mmean=mean)
mean=nbp_mean.to_array(dim='new').mean('new')*maskls.LSMASK; nbp_mean=nbp_mean.assign(mmean=mean)

Mt_nep=Mt_nep.assign_attrs(units="kgC m-2", description="Carbon mass calculated with NEP from TRENDY models", Ensamble="The mean value was calculated as the average of all models.")
Mt_nbp=Mt_nbp.assign_attrs(units="kgC m-2", description="Carbon mass calculated with NBP from TRENDY models", Ensamble="The mean value was calculated as the average of all models.")
cs_nep=cs_nep.assign_attrs(units="kgC m-2 yr", description="Carbon sequestration calculated with NEP from TRENDY models", Ensamble="The mean value was calculated as the average of all models.")
cs_nbp=cs_nbp.assign_attrs(units="kgC m-2 yr", description="Carbon sequestration calculated with NBP from TRENDY models", Ensamble="The mean value was calculated as the average of all models.")
gpp_mean=gpp_mean.assign_attrs(units="kgC m-2 yr-1", description="GPP from TRENDY models", Ensamble="The mean value was calculated as the average of all models.")
ra_mean=ra_mean.assign_attrs(units="kgC m-2 yr-1", description="Autotrophic respiration from TRENDY models", Ensamble="The mean value was calculated as the average of all models.")
rh_mean=rh_mean.assign_attrs(units="kgC m-2 yr-1", description="Heterotrophic respiration from TRENDY models", Ensamble="The mean value was calculated as the average of all models.")
nep_mean=nep_mean.assign_attrs(units="kgC m-2 yr-1", description="NEP calculated with GPP, ra and rh from TRENDY models", Ensamble="The mean value was calculated as the average of all models.")
nbp_mean=nbp_mean.assign_attrs(units="kgC m-2 yr-1", description="NBP from TRENDY models", Ensamble="The mean value was calculated as the average of all models.")


In [6]:
Mt_nep.to_netcdf('/home/data/ResultsCS/MT_nep_TRENDY.nc')
Mt_nbp.to_netcdf('/home/data/ResultsCS/MT_nbp_TRENDY.nc')

cs_nep.to_netcdf('/home/data/ResultsCS/CS_nep_TRENDY.nc')
cs_nbp.to_netcdf('/home/data/ResultsCS/CS_nbp_TRENDY.nc')

gpp_mean.to_netcdf('/home/data/ResultsCS/gpp_mean_TRENDY.nc')
ra_mean.to_netcdf('/home/data/ResultsCS/ra_mean_TRENDY.nc')
rh_mean.to_netcdf('/home/data/ResultsCS/rh_mean_TRENDY.nc')
nep_mean.to_netcdf('/home/data/ResultsCS/nep_mean_TRENDY.nc')
nbp_mean.to_netcdf('/home/data/ResultsCS/nbp_mean_TRENDY.nc')